# Feature management with Feast — the hands-on half

The practical companion to **`feast_slides.html`**. Two problems get solved here, and both of
them cost real money in production:

* **training/serving skew** — the features your model trained on are computed by different code
  from the features it is served, so they quietly disagree;
* **point-in-time correctness** — building a training set from "current" feature values leaks
  the future into the past, exactly the leak session 1 measured.

| Part | Deck slides | What you do |
|---|---|---|
| 0 · Setup | 11 | install, sandbox, generate an event history |
| 1 · Definitions | 12–14 | entities, sources, feature views, `feast apply` |
| 2 · Training set | 15 | `get_historical_features` — and **prove** the point-in-time join |
| 3 · Serving | 16 | `materialize`, then `get_online_features` in milliseconds |
| 4 · The same features both sides | 17 | train, serve, and show the values match |
| 5 · Advanced | 19–21 | on-demand transforms, TTL, feature services, registry |

Everything runs in a throwaway `feast_demo/` folder; the last cell tears the store down.

## Step 0.1 · Install

In [1]:
!pip install -q 'feast[redis]' pandas pyarrow scikit-learn matplotlib 2>/dev/null || pip install -q feast pandas pyarrow scikit-learn matplotlib

In [2]:
import feast, pandas as pd, numpy as np
print("feast  ", feast.__version__)
print("pandas ", pd.__version__)

/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


feast   0.40.1
pandas  2.2.3


## Step 0.2 · A sandbox to work in

In [3]:
import os, shutil, pathlib

BASE = pathlib.Path.cwd()          # the folder this notebook lives in
PROJ = BASE / "feast_demo"           # a throwaway sandbox, deleted by the last cell

if PROJ.exists():
    shutil.rmtree(PROJ)            # re-running this notebook is always safe
PROJ.mkdir(parents=True)
os.chdir(PROJ)
print("working inside:", os.getcwd())

working inside: /home/shamaseen/Desktop/Shai/qafza/free_Training/qafza-free-traning/09-feature-management/feast_demo


## Step 0.3 · An event history worth serving

A ride-sharing service. Every hour we compute per-driver statistics from completed trips. That
history — **feature values, each stamped with the time it became true** — is the raw material a
feature store manages.

The timestamp is the whole point. Without it there is no way to ask "what did we know about
driver 3 at 14:00 last Tuesday?", which is exactly what an honest training set requires.

In [4]:
import os
from datetime import datetime, timedelta

os.makedirs("driver_repo/data", exist_ok=True)
rng = np.random.default_rng(11)

DRIVERS = [1001, 1002, 1003, 1004, 1005]
START = datetime(2026, 3, 1)
HOURS = 24 * 10                       # ten days of hourly statistics

rows = []
level = {d: rng.uniform(.4, .9) for d in DRIVERS}      # each driver has a baseline
for h in range(HOURS):
    ts = START + timedelta(hours=h)
    for d in DRIVERS:
        level[d] = float(np.clip(level[d] + rng.normal(0, .02), .1, .99))   # drifts slowly
        rows.append({
            "driver_id": d,
            "event_timestamp": ts,
            "conv_rate": round(level[d], 4),
            "acc_rate": round(float(np.clip(level[d] + rng.normal(0, .05), .05, .99)), 4),
            "avg_daily_trips": int(rng.integers(2, 40)),
            "created": ts,                     # when the row was WRITTEN (vs became true)
        })

stats = pd.DataFrame(rows)

# Feast wants UTC timezone-AWARE timestamps. Naive ones survive the historical join
# (it localises them for you) but materialisation silently matches nothing, so the
# online store comes back full of None. This one line prevents a baffling afternoon.
stats["event_timestamp"] = stats.event_timestamp.dt.tz_localize("UTC")
stats["created"] = stats.created.dt.tz_localize("UTC")

stats.to_parquet("driver_repo/data/driver_stats.parquet", index=False)
print(f"{len(stats)} feature rows, {stats.driver_id.nunique()} drivers, "
      f"{stats.event_timestamp.min()} .. {stats.event_timestamp.max()}")
stats.head(3)

1200 feature rows, 5 drivers, 2026-03-01 00:00:00+00:00 .. 2026-03-10 23:00:00+00:00


,driver_id,event_timestamp,conv_rate,acc_rate,avg_daily_trips,created
0,1001,2026-03-01 00:00:00+00:00,0.4537,0.4822,22,2026-03-01 00:00:00+00:00
1,1002,2026-03-01 00:00:00+00:00,0.6646,0.5722,6,2026-03-01 00:00:00+00:00
2,1003,2026-03-01 00:00:00+00:00,0.7321,0.7273,18,2026-03-01 00:00:00+00:00


---
# Part 1 — Define the features once   ·   deck slides 12–14

Three concepts, and that is nearly all of Feast:

| Concept | Is | Here |
|---|---|---|
| **Entity** | the thing features describe, and its join key | a driver, keyed by `driver_id` |
| **Source** | where the feature history lives | the parquet file we just wrote |
| **FeatureView** | a named group of features, from a source, for an entity | `driver_hourly_stats` |

In [5]:
%%writefile driver_repo/feature_store.yaml
project: driver_ranking
provider: local
registry: data/registry.db
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3

Writing driver_repo/feature_store.yaml


In [6]:
%%writefile driver_repo/features.py
"""Feature definitions. This file is the contract -- training and serving both read it."""
from datetime import timedelta
from pathlib import Path

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

# An absolute path, derived from THIS file's location. Feast resolves a relative
# FileSource path against the working directory, not the repo -- so a notebook run
# from one directory up silently cannot find the data.
DATA = Path(__file__).parent / "data" / "driver_stats.parquet"

# WHAT the features describe, and the column used to join
driver = Entity(name="driver", join_keys=["driver_id"], description="a ride-sharing driver")

# WHERE the history lives. timestamp_field is what makes point-in-time joins possible.
driver_stats_source = FileSource(
    name="driver_stats_source",
    path=str(DATA),
    timestamp_field="event_timestamp",     # when the value became true
    created_timestamp_column="created",    # when we learned it (breaks ties)
)

# WHICH features, and for how long a value stays usable
driver_hourly_stats = FeatureView(
    name="driver_hourly_stats",
    entities=[driver],
    ttl=timedelta(days=3),                 # older than this = too stale to use
    schema=[
        Field(name="conv_rate",       dtype=Float32),
        Field(name="acc_rate",        dtype=Float32),
        Field(name="avg_daily_trips", dtype=Int64),
    ],
    source=driver_stats_source,
    online=True,                           # materialise for low-latency serving
    tags={"team": "marketplace", "owner": "ml-platform"},
)


Writing driver_repo/features.py


In [7]:
import subprocess
r = subprocess.run(["feast", "apply"], cwd="driver_repo", capture_output=True, text=True)
print(r.stdout[-1200:] or r.stderr[-1200:])
assert r.returncode == 0, "feast apply must succeed"

Created entity driver
Created feature view driver_hourly_stats

Created sqlite table driver_ranking_driver_hourly_stats




In [8]:
!cd driver_repo && feast entities list && echo && feast feature-views list

/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
NAME    DESCRIPTION            TYPE
driver  a ride-sharing driver  ValueType.UNKNOWN

/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/shamaseen/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (ve

`feast apply` did not move any data. It registered the *definitions* in the registry, so that
every consumer — a training job, an API, another team — resolves `driver_hourly_stats:conv_rate`
to the same thing. That shared registry is the feature store's real product.

---
# Part 2 — A training set, without leaking the future   ·   deck slides 15

Here is the mistake a feature store exists to prevent.

You have labels: "driver 1001 was selected at 2026-03-03 09:00". You need the features **as they
were at that moment**. The naive approach joins on `driver_id` and takes the current value —
which is the value from *after* the event. That is future information, and it inflates your
offline score exactly like the leaks in session 1.

In [9]:
from feast import FeatureStore

store = FeatureStore(repo_path="driver_repo")

# labels: an entity, a timestamp, an outcome
label_times = pd.to_datetime([
    "2026-03-03 09:00", "2026-03-05 14:00",
    "2026-03-07 18:00", "2026-03-09 06:00",
]).tz_localize("UTC")                     # tz-aware, to match the feature timestamps
entity_df = pd.DataFrame({
    "driver_id":       [1001, 1002, 1001, 1003],
    "event_timestamp": label_times,
    "selected":        [1, 0, 1, 1],
})
print(entity_df.to_string(index=False))

 driver_id           event_timestamp  selected
      1001 2026-03-03 09:00:00+00:00         1
      1002 2026-03-05 14:00:00+00:00         0
      1001 2026-03-07 18:00:00+00:00         1
      1003 2026-03-09 06:00:00+00:00         1


In [10]:
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
).to_df()

print(training_df.to_string(index=False))

 driver_id           event_timestamp  selected  conv_rate  acc_rate  avg_daily_trips
      1001 2026-03-03 09:00:00+00:00         1     0.3986    0.3709               36
      1002 2026-03-05 14:00:00+00:00         0     0.6971    0.7690               10
      1001 2026-03-07 18:00:00+00:00         1     0.4107    0.4191               39
      1003 2026-03-09 06:00:00+00:00         1     0.9495    0.8573               36


## Step 2.1 · Prove it is point-in-time correct

Do not take the join on trust. For the first label, look at the raw history around that
timestamp and check Feast handed us the **last value at or before** it, never a later one.

In [11]:
row = entity_df.iloc[0]
did, ts = int(row.driver_id), row.event_timestamp

window = stats[(stats.driver_id == did) &
               (stats.event_timestamp >= ts - timedelta(hours=2)) &
               (stats.event_timestamp <= ts + timedelta(hours=2))]
print(f"raw history for driver {did} around {ts}:")
print(window[["event_timestamp", "conv_rate"]].to_string(index=False))

expected = stats[(stats.driver_id == did) & (stats.event_timestamp <= ts)] \
             .sort_values("event_timestamp").iloc[-1]
served = training_df[(training_df.driver_id == did) &
                     (training_df.event_timestamp == ts)].iloc[0]

print(f"\nlast value at or before {ts}: {expected.conv_rate}  (at {expected.event_timestamp})")
print(f"what Feast returned:            {served.conv_rate:.4f}")
assert abs(float(served.conv_rate) - float(expected.conv_rate)) < 1e-6, \
    "the historical join must be point-in-time correct"
print("\nmatched -- no future value leaked into the training row.")

raw history for driver 1001 around 2026-03-03 09:00:00+00:00:
          event_timestamp  conv_rate
2026-03-03 07:00:00+00:00     0.3576
2026-03-03 08:00:00+00:00     0.3578
2026-03-03 09:00:00+00:00     0.3986
2026-03-03 10:00:00+00:00     0.3775
2026-03-03 11:00:00+00:00     0.3947

last value at or before 2026-03-03 09:00:00+00:00: 0.3986  (at 2026-03-03 09:00:00+00:00)
what Feast returned:            0.3986

matched -- no future value leaked into the training row.


In [12]:
# and what the naive join would have given us instead
latest = stats.sort_values("event_timestamp").groupby("driver_id").last().reset_index()
naive = entity_df.merge(latest[["driver_id", "conv_rate"]], on="driver_id", suffixes=("", "_now"))

cmp = training_df[["driver_id", "event_timestamp", "conv_rate"]].merge(
        naive[["driver_id", "conv_rate"]].rename(columns={"conv_rate": "naive_latest"}),
        on="driver_id")
cmp["difference"] = (cmp.naive_latest - cmp.conv_rate).round(4)
print(cmp.to_string(index=False))
print("\nEvery non-zero difference is future information the naive join would have leaked.")
assert cmp.difference.abs().sum() > 0, "the demo needs the naive join to differ"

 driver_id           event_timestamp  conv_rate  naive_latest  difference
      1001 2026-03-03 09:00:00+00:00     0.3986        0.1500     -0.2486
      1001 2026-03-03 09:00:00+00:00     0.3986        0.1500     -0.2486
      1002 2026-03-05 14:00:00+00:00     0.6971        0.6941     -0.0030
      1001 2026-03-07 18:00:00+00:00     0.4107        0.1500     -0.2607
      1001 2026-03-07 18:00:00+00:00     0.4107        0.1500     -0.2607
      1003 2026-03-09 06:00:00+00:00     0.9495        0.9025     -0.0470

Every non-zero difference is future information the naive join would have leaked.


---
# Part 3 — Serving the same features, fast   ·   deck slides 16

`get_historical_features` reads parquet. That is fine for training and hopeless for an API that
must answer in milliseconds.

**Materialisation** copies the latest value per entity into the online store, so serving is a
key lookup.

In [13]:
import subprocess

start = (stats.event_timestamp.min() - timedelta(minutes=1)).isoformat()
end   = (stats.event_timestamp.max() + timedelta(minutes=1)).isoformat()

# An EXPLICIT window. `feast materialize-incremental <end>` is the usual command, but on a
# first run it derives its own start from the feature view TTL, and it is easy to end up
# materialising a window your data does not cover -- which shows up much later as an online
# store full of None rather than as an error here.
r = subprocess.run(["feast", "materialize", start, end],
                   cwd="driver_repo", capture_output=True, text=True)
print((r.stdout or r.stderr)[-700:])
assert r.returncode == 0, "materialisation must succeed"


Materializing 1 feature views from 2026-03-01 02:59:00+03:00 to 2026-03-11 02:01:00+03:00 into the sqlite online store.

driver_hourly_stats:



In [14]:
import time

t0 = time.perf_counter()
online = store.get_online_features(
    features=["driver_hourly_stats:conv_rate",
              "driver_hourly_stats:acc_rate",
              "driver_hourly_stats:avg_daily_trips"],
    entity_rows=[{"driver_id": 1001}, {"driver_id": 1002}, {"driver_id": 1003}],
).to_dict()
dt = (time.perf_counter() - t0) * 1000

print(pd.DataFrame(online).to_string(index=False))
print(f"\nonline lookup for 3 drivers: {dt:.2f} ms")

 driver_id  avg_daily_trips  conv_rate  acc_rate
      1001                2     0.1500    0.1335
      1002               30     0.6941    0.6799
      1003               12     0.9025    0.9596

online lookup for 3 drivers: 3.34 ms


In [15]:
# measure it properly -- one row, the shape an API actually serves
lat = []
for _ in range(200):
    t0 = time.perf_counter()
    store.get_online_features(features=["driver_hourly_stats:conv_rate"],
                              entity_rows=[{"driver_id": 1001}]).to_dict()
    lat.append((time.perf_counter() - t0) * 1000)
lat.sort()
p50, p95 = lat[len(lat)//2], lat[int(len(lat)*.95)]
print(f"single-entity online lookup:  p50 {p50:.2f} ms   p95 {p95:.2f} ms")
print("\nsqlite locally; Redis or DynamoDB in production, same call from your code.")

single-entity online lookup:  p50 0.40 ms   p95 0.54 ms

sqlite locally; Redis or DynamoDB in production, same call from your code.


> **The unknown entity question.** Ask for a driver that does not exist and Feast returns
> `None` rather than raising — your API has to decide what to do about that. Returning a default
> silently is how a model ends up scoring every new user as average.

In [16]:
unknown = store.get_online_features(
    features=["driver_hourly_stats:conv_rate"],
    entity_rows=[{"driver_id": 999999}]).to_dict()
print("unknown driver ->", unknown)
print("\nNone, not an exception. Handle it explicitly in the serving path.")

unknown driver -> {'driver_id': [999999], 'conv_rate': [None]}

None, not an exception. Handle it explicitly in the serving path.


---
# Part 4 — The same features on both sides   ·   deck slides 17

This is the payoff. Train on `get_historical_features`, serve from `get_online_features`, and the
feature values agree **because they came from one definition**.

In [17]:
from sklearn.linear_model import LogisticRegression

# a slightly larger training set, built the correct way
rng = np.random.default_rng(3)
n = 240
ids = rng.choice(DRIVERS, n)
times = pd.to_datetime([START + timedelta(hours=int(h))
                        for h in rng.integers(24, HOURS, n)]).tz_localize("UTC")
big_entity = pd.DataFrame({"driver_id": ids, "event_timestamp": times})

train = store.get_historical_features(
    entity_df=big_entity,
    features=["driver_hourly_stats:conv_rate", "driver_hourly_stats:acc_rate",
              "driver_hourly_stats:avg_daily_trips"]).to_df().dropna()

FEATURES = ["conv_rate", "acc_rate", "avg_daily_trips"]
train["selected"] = (train.conv_rate * 2 + train.acc_rate
                     + rng.normal(0, .3, len(train)) > 1.6).astype(int)

model = LogisticRegression(max_iter=1000).fit(train[FEATURES], train.selected)
print(f"trained on {len(train)} point-in-time correct rows")
print("coefficients:", dict(zip(FEATURES, model.coef_[0].round(3))))

trained on 209 point-in-time correct rows
coefficients: {'conv_rate': 4.332, 'acc_rate': 4.231, 'avg_daily_trips': 0.019}


In [18]:
# serve: fetch from the ONLINE store and predict, with no feature code of our own
def score(driver_id: int):
    f = store.get_online_features(
        features=[f"driver_hourly_stats:{c}" for c in FEATURES],
        entity_rows=[{"driver_id": driver_id}]).to_dict()
    row = pd.DataFrame({c: f[c] for c in FEATURES})
    return float(model.predict_proba(row)[0, 1]), row.iloc[0].to_dict()

for d in DRIVERS:
    p, feats = score(d)
    print(f"driver {d}: p(selected)={p:.3f}   features={feats}")

driver 1001: p(selected)=0.029   features={'conv_rate': 0.15000000596046448, 'acc_rate': 0.13349999487400055, 'avg_daily_trips': 2.0}
driver 1002: p(selected)=0.842   features={'conv_rate': 0.694100022315979, 'acc_rate': 0.6798999905586243, 'avg_daily_trips': 30.0}
driver 1003: p(selected)=0.968   features={'conv_rate': 0.9024999737739563, 'acc_rate': 0.9595999717712402, 'avg_daily_trips': 12.0}
driver 1004: p(selected)=0.444   features={'conv_rate': 0.45190000534057617, 'acc_rate': 0.45590001344680786, 'avg_daily_trips': 35.0}
driver 1005: p(selected)=0.074   features={'conv_rate': 0.1851000040769577, 'acc_rate': 0.24230000376701355, 'avg_daily_trips': 22.0}


In [19]:
# and prove the two paths agree on the values themselves
offline_latest = store.get_historical_features(
    entity_df=pd.DataFrame({"driver_id": DRIVERS,
                            "event_timestamp": [stats.event_timestamp.max()] * len(DRIVERS)}),
    features=[f"driver_hourly_stats:{c}" for c in FEATURES]).to_df()

online_all = pd.DataFrame(store.get_online_features(
    features=[f"driver_hourly_stats:{c}" for c in FEATURES],
    entity_rows=[{"driver_id": d} for d in DRIVERS]).to_dict())

merged = offline_latest[["driver_id"] + FEATURES].merge(
    online_all, on="driver_id", suffixes=("_offline", "_online"))
merged["conv_diff"] = (merged.conv_rate_offline - merged.conv_rate_online).abs().round(6)
print(merged[["driver_id", "conv_rate_offline", "conv_rate_online", "conv_diff"]].to_string(index=False))
assert merged.conv_diff.max() < 1e-5, "offline and online must agree -- otherwise you have skew"
print("\nOffline and online agree to 5 decimal places. That is training/serving skew, eliminated.")

 driver_id  conv_rate_offline  conv_rate_online  conv_diff
      1001             0.1500            0.1500        0.0
      1004             0.4519            0.4519        0.0
      1003             0.9025            0.9025        0.0
      1002             0.6941            0.6941        0.0
      1005             0.1851            0.1851        0.0

Offline and online agree to 5 decimal places. That is training/serving skew, eliminated.


---
# Part 5 — Advanced   ·   deck slides 19–21

## Step 5.1 · On-demand feature views

Some features can only be computed at request time, from the request itself — a ratio involving
a value the caller sends. An **on-demand** view keeps that transformation in the same registry,
so training and serving still share one definition.

In [20]:
%%writefile driver_repo/on_demand.py
"""A feature computed at request time, from stored features plus request input."""
import pandas as pd

from feast import Field, RequestSource
# NOTE: `from feast import on_demand_feature_view` imports the MODULE of that name,
# not the decorator, and calling it raises "TypeError: 'module' object is not callable".
from feast.on_demand_feature_view import on_demand_feature_view
from feast.types import Float64, Int64

from features import driver_hourly_stats

# values that arrive WITH the request, not from the store
request_source = RequestSource(
    name="trip_request",
    schema=[Field(name="trip_distance_km", dtype=Float64)],
)


@on_demand_feature_view(
    sources=[driver_hourly_stats, request_source],
    schema=[Field(name="trips_per_km", dtype=Float64),
            Field(name="is_long_trip", dtype=Int64)],
    mode="pandas",          # be explicit: without it 0.40 raises
)                           # "Couldn't infer value type from empty value" during apply
def trip_features(inputs: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    km = inputs["trip_distance_km"].clip(lower=0.1)
    out["trips_per_km"] = inputs["avg_daily_trips"] / km
    out["is_long_trip"] = (inputs["trip_distance_km"] > 15).astype("int64")
    return out


Writing driver_repo/on_demand.py


In [21]:
r = subprocess.run(["feast", "apply"], cwd="driver_repo", capture_output=True, text=True)
print(r.stdout[-800:] or "")
if r.returncode != 0:
    print("STDERR:", r.stderr[-800:])
assert r.returncode == 0, "feast apply must succeed after adding the on-demand view"

# re-open the store so it sees the new registry contents
store = FeatureStore(repo_path="driver_repo")

odfv = store.get_online_features(
    features=["driver_hourly_stats:avg_daily_trips",
              "trip_features:trips_per_km", "trip_features:is_long_trip"],
    entity_rows=[{"driver_id": 1001, "trip_distance_km": 22.0},
                 {"driver_id": 1002, "trip_distance_km": 3.5}],
).to_dict()
print(pd.DataFrame(odfv).to_string(index=False))
print("\ntrips_per_km was computed at request time -- from a stored feature and a request field.")


Created on demand feature view trip_features

No changes to infrastructure

 driver_id  avg_daily_trips  trips_per_km  is_long_trip
      1001                2      0.090909             1
      1002               30      8.571429             0

trips_per_km was computed at request time -- from a stored feature and a request field.


## Step 5.2 · Feature services — a named bundle per model

A model does not want "all our features". It wants the specific set it was trained on. A
`FeatureService` names that set, so the serving code asks for one thing and cannot drift.

```python
from feast import FeatureService

driver_ranking_v1 = FeatureService(
    name="driver_ranking_v1",
    features=[driver_hourly_stats[["conv_rate", "acc_rate"]], trip_features],
)

# then, in serving:
store.get_online_features(features=store.get_feature_service("driver_ranking_v1"),
                         entity_rows=rows)
```

Version the service, not just the model: `driver_ranking_v2` can add a feature while v1 keeps
answering for the model still in production.

## Step 5.3 · TTL, and what it actually does

`ttl=timedelta(days=3)` on the feature view means: when building a training set, a feature value
more than three days older than the label is treated as **missing**, not carried forward.

* Too short → training rows full of nulls.
* Too long → you train on values that were badly stale by then.
* It is a statement about how fast your features go out of date, so pick it per feature view.

Materialisation is bounded by it too, which is why `materialize-incremental` takes an end
timestamp rather than "now".

In [22]:
# a label far older than any feature value, to see TTL bite
old = pd.DataFrame({"driver_id": [1001],
                    "event_timestamp": pd.to_datetime([START - timedelta(days=5)]).tz_localize("UTC")})
res = store.get_historical_features(
    entity_df=old, features=["driver_hourly_stats:conv_rate"]).to_df()
print(res.to_string(index=False))
print("\nNaN -- no feature value existed within the TTL of that label. Correct, and honest.")

Empty DataFrame
Columns: [driver_id, event_timestamp, conv_rate]
Index: []

NaN -- no feature value existed within the TTL of that label. Correct, and honest.


---
# Reference — worth knowing, not demonstrated here

| Topic | One-line version |
|---|---|
| Real online stores | Redis, DynamoDB, Bigtable — change `feature_store.yaml`, not your code |
| Real offline stores | BigQuery, Snowflake, Redshift — the historical join runs as SQL there |
| Stream sources | Kafka/Kinesis push sources for features that must be seconds fresh |
| Scheduling materialisation | a cron or Airflow job running `materialize-incremental` |
| `feast plan` | see what `apply` would change before it changes it |
| Permissions & registry sharing | one registry, many teams — that is the point of a store |
| Feast vs building it yourself | you can build this; you will end up rebuilding point-in-time joins |

## Recap

| You wanted to… | Do this |
|---|---|
| describe features once | `features.py` + `feast apply` |
| an honest training set | `get_historical_features(entity_df with timestamps)` |
| avoid leaking the future | never join on "current" values — that is what the timestamp is for |
| serve in milliseconds | `materialize-incremental`, then `get_online_features` |
| stop training/serving skew | both sides read the same feature view |
| features from the request | an `@on_demand_feature_view` |
| a stable set per model | a `FeatureService`, versioned |
| control staleness | `ttl` on the feature view |

## What to do at work tomorrow

1. Find one feature that is computed twice — once in a training notebook, once in serving code.
   That pair is your skew, and it is probably already costing you accuracy.
2. Write down whether your training set was built with point-in-time joins. If nobody knows, it
   was not.
3. Define that one feature in a feature view and read it from both sides.
4. Only then worry about Redis, streaming, or a second team.

## Cleanup

Tears the store down and removes the sandbox.

In [23]:
import subprocess, shutil, os
subprocess.run(["feast", "teardown"], cwd="driver_repo", capture_output=True, text=True)
print("feature store torn down")
os.chdir(BASE)
shutil.rmtree(PROJ, ignore_errors=True)
print("sandbox removed")

feature store torn down
sandbox removed
